In [81]:
import pandas as pd
from datetime import datetime as dt , timedelta
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors

In [82]:
data=pd.read_csv(r"C:\Users\saad\Desktop\PyProjects\online_retail.csv")
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [83]:
data.tail()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [84]:
data.dropna(subset=['CustomerID'],inplace=True)

In [85]:
data['InvoiceDate']=pd.to_datetime(data['InvoiceDate'])
data['TotalAmount']=data['Quantity']*data['UnitPrice']


In [86]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [87]:
reference_date=pd.Timestamp(dt.now().date())


In [88]:
reference_date=data['InvoiceDate'].max() + timedelta(days=1)

In [89]:
reference_date

Timestamp('2011-12-10 12:50:00')

In [90]:
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalAmount': 'sum' 
})

In [91]:
rfm.rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalAmount': 'Value'}, inplace=True)
rfm.head()

,Recency,Frequency,Value
CustomerID,,,
12346.0,326,2,0.00
12347.0,2,182,4310.00
12348.0,75,31,1797.24
12349.0,19,73,1757.55
12350.0,310,17,334.40


In [92]:
quantiles = rfm.quantile(q=[0.25, 0.5, 0.75])
def Rscore(x,p,d):
    if p == 'Recency':
        if x <= d[p][0.25]:
            return 4
        elif x <= d[p][0.50]:
            return 3
        elif x <= d[p][0.75]: 
            return 2
        else:
            return 1
    else:
        if x <= d[p][0.25]:
            return 1
        elif x <= d[p][0.50]:
            return 2
        elif x <= d[p][0.75]: 
            return 3
        else:
            return 4

rfm['R']=rfm['Recency'].apply(Rscore, args=('Recency',quantiles,))
rfm['F']=rfm['Frequency'].apply(Rscore, args=('Frequency',quantiles,))
rfm['M']=rfm['Value'].apply(Rscore, args=('Value',quantiles,))

rfm.head()

,Recency,Frequency,Value,R,F,M
CustomerID,,,,,,
12346.0,326,2,0.00,1,1,1
12347.0,2,182,4310.00,4,4,4
12348.0,75,31,1797.24,2,2,4
12349.0,19,73,1757.55,3,3,4
12350.0,310,17,334.40,1,1,2


In [93]:
rfm['RFM_segment']=rfm['R'].astype(str)+rfm['F'].astype(str)+rfm['M'].astype(str)
rfm['RFM_score']=rfm[['R','F','M']].sum(axis=1)

In [94]:
rfm.head()

,Recency,Frequency,Value,R,F,M,RFM_segment,RFM_score
CustomerID,,,,,,,,
12346.0,326,2,0.00,1,1,1,111,3
12347.0,2,182,4310.00,4,4,4,444,12
12348.0,75,31,1797.24,2,2,4,224,8
12349.0,19,73,1757.55,3,3,4,334,10
12350.0,310,17,334.40,1,1,2,112,4


In [95]:
segment_labels = ['Low-Value', 'Mid-Value', 'High-Value']

def assign_segment(score):
    if score < 5:
        return segment_labels[0]
    elif score < 9:
        return segment_labels[1]
    else:
        return segment_labels[2]

rfm['RFM_segment_Label']=rfm['RFM_score'].apply(assign_segment)

rfm.head()

,Recency,Frequency,Value,R,F,M,RFM_segment,RFM_score,RFM_segment_Label
CustomerID,,,,,,,,,
12346.0,326,2,0.00,1,1,1,111,3,Low-Value
12347.0,2,182,4310.00,4,4,4,444,12,High-Value
12348.0,75,31,1797.24,2,2,4,224,8,Mid-Value
12349.0,19,73,1757.55,3,3,4,334,10,High-Value
12350.0,310,17,334.40,1,1,2,112,4,Low-Value


In [96]:
segment_counts = rfm['RFM_segment_Label'].value_counts().reset_index()
segment_counts.columns = ['RFM_segment', 'count']
segment_counts=segment_counts.sort_values(by='RFM_segment')

In [97]:
fig=px.bar(segment_counts, 
           x='RFM_segment', 
           y='count', 
           title='Customer Distribution by RFM Segment',
           labels={'RFM_segment':'RFM Segment', 'count':'Number of Customers'},
           color='RFM_segment',
           color_discrete_sequence=px.colors.qualitative.Pastel)


fig.show()

In [98]:
rfm['RFM_Customer_Segments']=''

rfm.loc[rfm['RFM_score']>=9,'RFM_Customer_Segments']='VIP/Loyal'
rfm.loc[(rfm['RFM_score']>=6) & (rfm['RFM_score']<9),'RFM_Customer_Segments']='Potential Loyalist'
rfm.loc[(rfm['RFM_score']>=5) & (rfm['RFM_score']<6),'RFM_Customer_Segments']='At Risk Customers'
rfm.loc[(rfm['RFM_score']>=4) & (rfm['RFM_score']<5),'RFM_Customer_Segments']="Can't Lose"
rfm.loc[(rfm['RFM_score']>=3) & (rfm['RFM_score']<4),'RFM_Customer_Segments']='Lost'
segment_counts = rfm['RFM_Customer_Segments'].value_counts().reset_index()

In [99]:
segments_product_counts = rfm.groupby(['RFM_segment_Label','RFM_Customer_Segments']).size().reset_index(name='count')
segments_product_counts=segments_product_counts.sort_values('count',ascending=False) 

In [100]:
fig_treemap_segmet_product =px.treemap(segments_product_counts,
                                        path=['RFM_segment_Label','RFM_Customer_Segments'], 
                                        values='count',
                                        color='RFM_segment_Label',
                                        color_discrete_sequence=px.colors.qualitative.Pastel,
                                        title='RFM Customer Segments by Value')

fig_treemap_segmet_product.show()

In [101]:
vip_segment = rfm[rfm['RFM_Customer_Segments'] == 'VIP/Loyal']

In [102]:
fig = go.Figure()
fig.add_trace(go.Box(y=vip_segment['Recency'],name='Revency'))
fig.add_trace(go.Box(y=vip_segment['Frequency'],name='Frequency'))
fig.add_trace(go.Box(y=vip_segment['Value'],name='Value'))

In [103]:
correlation_matrix =vip_segment[['Recency', 'Frequency', 'Value']].corr()

In [104]:
fig_heatmap = go.Figure(data=go.Heatmap(
                    z=correlation_matrix.values,
                    x=correlation_matrix.columns,
                    y=correlation_matrix.columns,
                    colorscale='RdBu',
                    colorbar=dict(title='Correlation')) 
)

fig_heatmap.update_layout(title='Correlation Matrix of RFM Values within Champions Segment')

# Display the heatmap
fig_heatmap.show()